<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/Multi_Step_Forecasting_Strategies.ipynb)

# Tomorrow, Three Ways: Recursive, Direct, Multi-Output
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

The capstone forecast 24 hours at once with `Dense(24)`. That is one of **three** ways to get from a one-step model to a whole day, and the one most people try first is the one with a trap in it:

| strategy | how | the catch |
| :-- | :-- | :-- |
| **Recursive** (autoregressive) | one model, one hour ahead; feed the prediction back in as "last hour" and roll forward 24 times | your own errors become tomorrow's inputs - and the covariates at future hours have to come from *somewhere* |
| **Direct** | one model per horizon: a 1-hour model, a 2-hour model, ... a 24-hour model | 24 models to train and maintain; nothing ties the horizons together |
| **Multi-output** (MIMO) | one model, `Dense(24)` - the capstone | one shot, no rolling; the model has to learn the whole daily shape at once |

You'll build all three on the same data and plot **error by horizon** for each. Then the question that trips everyone up in practice: when past demand is a feature, *what goes in that column for the hours you haven't seen yet?*

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 18 — Tomorrow, three ways: recursive, direct, multi-output
- Setup: the capstone's multivariate one-step LSTM (last 24 hours -> next hour, demand's own past in the window). The question: how do you get 24 hours out of it?
- RECURSIVE: predict hour 1, write it into the demand column, slide the window, predict hour 2... Draw the window sliding on screen. The trap: after a few steps the window is mostly your own guesses.
- The covariate problem is the point of the video: for the future rows you need the clock (known - free), the weather (NOT known - use a forecast; here we cheat with the actuals and CALL it perfect foresight, then show the honest version with the weather frozen), and past demand (your own predictions).
- Exposure bias: the model trained on TRUE past demand, then runs on its own predictions. Errors compound - read the curve: 33 MW at hour 1 (best of everything), 331 at hour 24 with perfect weather, 414 with the weather frozen - WORSE than seasonal naive (227) from about hour 8 on.
- DIRECT: retrain the same model with the target shifted h hours out - a separate model per horizon (we do 1, 6, 12, 24: 42 / 142 / 178 / 207 MW). No rolling, no fake inputs; the cost is 24 models.
- MULTI-OUTPUT: Dense(24), the capstone. One model, one shot: 87 / 139 / 171 / 206 MW - weak at hour 1 (it's learning the whole day at once), tied with direct by hour 24, and both beat seasonal naive. Say which you'd ship and why (direct/MIMO for ops; recursive only for the first few hours).
- Real life: the weather column at future hours comes from the NWS forecast (we have a pipeline for that in class) - and the forecast's error becomes yours.
-->


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from keras.models import Sequential, load_model
from keras.layers import LSTM, Dense
from keras.callbacks import EarlyStopping
import keras
keras.utils.set_random_seed(5509)   # reproducibility: same numbers every run (CPU exact; a GPU may drift a little)

## Data and the one-step engine

Same prep as the capstone. The one-step LSTM is the engine every strategy starts from.

<!-- WINDOW-FN -->
## The window-making function: `split_sequences`

Target in the **last column**, then:

- **`n_steps`** is the **look-back**: X is the last `n_steps` rows of **every column, the target included**. Yesterday's demand is the most useful input there is.
- y is the target **`ahead` steps (1 = the very next step)** after the window ends.

Result: X is `(samples, n_steps, n_features)`.


In [ ]:
# same prep as the demand capstone: sort, fill, clock features, Demand LAST, train 2017-18 / test 2019
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/main/OPIM5509_Module4_Files/data/BDL_cleanweather_energy.csv"
df = pd.read_csv(url, parse_dates=["Datetime"]).sort_values("Datetime").set_index("Datetime").ffill()

def add_clock(d):
    d = d.copy()
    d["hour_sin"] = np.sin(2*np.pi*d.index.hour/24);      d["hour_cos"] = np.cos(2*np.pi*d.index.hour/24)
    d["dow_sin"]  = np.sin(2*np.pi*d.index.dayofweek/7);  d["dow_cos"]  = np.cos(2*np.pi*d.index.dayofweek/7)
    return d

feats = ["BDL_tmpf", "BDL_dwpf", "BDL_relh", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "Demand"]   # Demand LAST
data  = add_clock(df.loc["2017-01-01":"2019-12-31"])[feats]
train, test = data.loc[:"2018-12-31"], data.loc["2019-01-01":]

sc   = MinMaxScaler().fit(train)               # fit on TRAIN only
sc_y = MinMaxScaler().fit(train[["Demand"]])   # target-only scaler to get MW back
tr_s, te_s = sc.transform(train), sc.transform(test)

def split_sequences(seqs, n_steps, ahead=1):
    """inputs = the past n_steps rows (every column); target = Demand `ahead` steps after the window ends"""
    X, y = [], []
    for i in range(len(seqs) - n_steps - ahead + 1):
        X.append(seqs[i:i+n_steps, :]); y.append(seqs[i+n_steps+ahead-1, -1])
    return np.array(X), np.array(y)

n_steps = 24
X_train, y_train = split_sequences(tr_s, n_steps)
X_test,  y_test  = split_sequences(te_s, n_steps)
n_features = X_train.shape[2]
y_true = sc_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=0)
print("train:", X_train.shape, "| test:", X_test.shape)

In [ ]:
one_step = Sequential([LSTM(32, input_shape=(n_steps, n_features)), Dense(1)])
one_step.compile(optimizer="adam", loss="mse", metrics=["mae"])
one_step.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
pred_1h = sc_y.inverse_transform(one_step.predict(X_test, verbose=0)).ravel()
print(f"one-step LSTM, 1 hour ahead: MAE {mean_absolute_error(y_true, pred_1h):.1f} MW")

## Strategy 1: recursive (autoregressive)

Take the last 24 real hours, predict hour +1. Now build the row for hour +1: the **clock** we know, the **weather** we... don't (more on that below), and **demand** is the number we just predicted. Slide the window forward one row and predict hour +2. Repeat 24 times.

The model was trained where the demand column was always *true* history. At forecast time it sees its own guesses in that column - that mismatch has a name, **exposure bias** - and every error it makes becomes part of tomorrow's input.

In [ ]:
def recursive_forecast(model, seqs, n_steps, horizon, weather_cols, mode="actual"):
    """Roll a one-step model horizon steps ahead from EVERY window in seqs.
    mode='actual'  -> future weather rows come from the real future (perfect foresight - a cheat, labelled as such)
    mode='persist' -> future weather is frozen at the last observed hour (what you'd have with no forecast)"""
    n_orig = len(seqs) - n_steps - horizon + 1
    window = np.stack([seqs[i:i+n_steps] for i in range(n_orig)])          # (origins, 24, 8) - all real history
    preds = np.zeros((n_orig, horizon))
    for h in range(horizon):
        p = model.predict(window, verbose=0).ravel()                          # predict the next hour for every origin at once
        preds[:, h] = p
        future = np.stack([seqs[i+n_steps+h] for i in range(n_orig)]).copy() # the real row for hour +h+1 (we only take the clock from it)
        if mode == "persist":
            future[:, weather_cols] = window[:, -1, weather_cols]            # no forecast: freeze the weather
        future[:, -1] = p                                                    # the demand column is OUR PREDICTION now
        window = np.concatenate([window[:, 1:, :], future[:, None, :]], axis=1)   # slide the window
    return preds

horizon = 24
weather_cols = [feats.index(c) for c in ["BDL_tmpf", "BDL_dwpf", "BDL_relh"]]
rec_actual  = recursive_forecast(one_step, te_s, n_steps, horizon, weather_cols, mode="actual")
rec_persist = recursive_forecast(one_step, te_s, n_steps, horizon, weather_cols, mode="persist")

# truth for the same origins and horizons, in MW
n_orig = rec_actual.shape[0]
truth = np.stack([te_s[i+n_steps:i+n_steps+horizon, -1] for i in range(n_orig)])
to_mw = lambda a: sc_y.inverse_transform(a.reshape(-1, 1)).reshape(a.shape)
truth_mw, rec_actual_mw, rec_persist_mw = to_mw(truth), to_mw(rec_actual), to_mw(rec_persist)

mae_rec_actual  = np.abs(rec_actual_mw  - truth_mw).mean(axis=0)
mae_rec_persist = np.abs(rec_persist_mw - truth_mw).mean(axis=0)
print("recursive, perfect-foresight weather - MAE at h=1, 6, 12, 24:", mae_rec_actual[[0, 5, 11, 23]].round(1))
print("recursive, weather frozen          - MAE at h=1, 6, 12, 24:", mae_rec_persist[[0, 5, 11, 23]].round(1))

## Strategy 2: direct (one model per horizon)

Same window, but the target is demand **h hours after the window ends**. Nothing is fed back, so there is no exposure bias - the model at horizon 12 learned, from real data, what 12 hours of drift looks like. The price: one model per horizon. We train four (1, 6, 12, 24) to see the curve; production would train all 24.

In [ ]:
direct_h, direct_mae = [1, 6, 12, 24], []
for h in direct_h:
    Xh_tr, yh_tr = split_sequences(tr_s, n_steps, ahead=h)
    Xh_te, yh_te = split_sequences(te_s, n_steps, ahead=h)
    m = Sequential([LSTM(32, input_shape=(n_steps, n_features)), Dense(1)])
    m.compile(optimizer="adam", loss="mse")
    m.fit(Xh_tr, yh_tr, epochs=25, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
    # score on the same origins as the recursive run (first n_orig windows)
    p = sc_y.inverse_transform(m.predict(Xh_te[:n_orig], verbose=0)).ravel()
    t = sc_y.inverse_transform(yh_te[:n_orig].reshape(-1, 1)).ravel()
    direct_mae.append(mean_absolute_error(t, p))
    print(f"direct model, {h:2d} hours ahead: MAE {direct_mae[-1]:.1f} MW")

## Strategy 3: multi-output (`Dense(24)`)

The capstone's approach: one model, 24 outputs, no rolling. Same 24-hour window so the comparison is fair.

<!-- WINDOW-FN -->
## The window-making function: `split_multistep`

For forecasting **several steps at once**:

- **`lookback`**: how many past rows go in (X is `(samples, lookback, n_features)`).
- **`horizon`**: how many future values come out (y is `(samples, horizon)`), one output per step, so the model ends in `Dense(horizon)`.

With `lookback=168, horizon=24`: the past week in, the next day out.


In [ ]:
def split_multistep(seqs, lookback, horizon):
    X, y = [], []
    for i in range(len(seqs) - lookback - horizon + 1):
        X.append(seqs[i:i+lookback, :]); y.append(seqs[i+lookback:i+lookback+horizon, -1])
    return np.array(X), np.array(y)

Xm_tr, ym_tr = split_multistep(tr_s, n_steps, horizon)
Xm_te, ym_te = split_multistep(te_s, n_steps, horizon)
mimo = Sequential([LSTM(32, input_shape=(n_steps, n_features)), Dense(horizon)])
mimo.compile(optimizer="adam", loss="mse", metrics=["mae"])
mimo.fit(Xm_tr, ym_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
mimo_mw = to_mw(mimo.predict(Xm_te, verbose=0))
mae_mimo = np.abs(mimo_mw - to_mw(ym_te)).mean(axis=0)
print("multi-output - MAE at h=1, 6, 12, 24:", mae_mimo[[0, 5, 11, 23]].round(1))

## Error by horizon - all three, against seasonal naive

In [ ]:
dem = test["Demand"].values
naive = np.array([mean_absolute_error(dem[n_steps+h:n_steps+h+n_orig], dem[n_steps+h-24:n_steps+h-24+n_orig]) for h in range(horizon)])

hrs = np.arange(1, horizon+1)
plt.figure(figsize=(9, 4))
plt.plot(hrs, mae_rec_persist, marker=".", label="recursive - weather frozen (honest)")
plt.plot(hrs, mae_rec_actual,  marker=".", label="recursive - perfect-foresight weather (cheat)")
plt.plot(hrs, mae_mimo,        marker=".", label="multi-output Dense(24)")
plt.plot(direct_h, direct_mae, "s", ms=8, label="direct (one model per horizon)")
plt.plot(hrs, naive, "--", color="gray", label="seasonal naive")
plt.xlabel("hours ahead"); plt.ylabel("test MAE (MW)"); plt.title("Three ways to forecast tomorrow"); plt.legend(); plt.show()

pd.DataFrame({"recursive (frozen wx)": mae_rec_persist[[0, 5, 11, 23]], "recursive (perfect wx)": mae_rec_actual[[0, 5, 11, 23]],
              "direct": direct_mae, "multi-output": mae_mimo[[0, 5, 11, 23]], "seasonal naive": naive[[0, 5, 11, 23]]},
             index=[f"h={h}" for h in direct_h]).round(1)

In [ ]:
# one origin, the three forecasts and reality
o = 24*200 + 6      # a day in mid-July, forecast made at 6 AM
plt.figure(figsize=(9, 3.5))
plt.plot(hrs, truth_mw[o], "k", lw=2, label="actual")
plt.plot(hrs, rec_persist_mw[o], label="recursive (frozen wx)"); plt.plot(hrs, rec_actual_mw[o], label="recursive (perfect wx)")
plt.plot(hrs, mimo_mw[o], label="multi-output")
plt.xlabel("hours ahead"); plt.ylabel("MW"); plt.title(f"Forecast issued {test.index[o+n_steps-1]}"); plt.legend(); plt.show()

### The same forecasts as time series

One day tells you little. Line the forecasts up by the hour they were **for**, and you can watch each strategy follow (or miss) the daily cycle and the heat waves. Each panel fixes the lead time: the top shows forecasts made **1 hour** earlier, the bottom forecasts made **24 hours** earlier, over two weeks in July.

In [ ]:
# the same forecasts as time series: line each forecast up with the hour it was FOR
w0, w1 = 24*181, 24*195                                  # two weeks in July 2019
fig, ax = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, sharey=True)
for a, h in zip(ax, [1, 24]):
    when = test.index[n_steps + h - 1 + np.arange(w0, w1)]       # the hour each forecast is for
    a.plot(when, truth_mw[w0:w1, h-1], color="black", lw=1.6, label="actual")
    a.plot(when, rec_persist_mw[w0:w1, h-1], lw=1, label="recursive, weather frozen")
    a.plot(when, rec_actual_mw[w0:w1, h-1], lw=1, ls="--", label="recursive, perfect weather")
    a.plot(when, mimo_mw[w0:w1, h-1], lw=1, label="multi-output Dense(24)")
    a.set_title(f"forecasts made {h} hour{'s' if h > 1 else ''} earlier"); a.set_ylabel("MW")
ax[0].legend(ncol=4, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

**Read it like a dispatcher.** One hour ahead, every strategy sits on the black line. A day ahead, both recursive forecasts **overshoot** the afternoon peaks (each step's error feeds the next one), and the frozen-weather version overshoots the most. The multi-output model keeps the daily shape and usually sits a little **low**.

## Where do the future covariates come from?

The recursive strategy forced the question; it applies to *every* strategy that uses covariates at the target hour. Three kinds of column:

| column | at hour +h you have... | what to do |
| :-- | :-- | :-- |
| **calendar / clock** (hour, weekday, holiday) | the exact value - it's a calendar | use it; it's free and it's the strongest covariate after demand itself |
| **weather** (temperature, dew point, humidity) | *not* the actual - only a **forecast** | feed the NWS forecast for hour +h; the forecast's error becomes your error, so validate with *archived forecasts*, not actuals (the "perfect foresight" curve above is the optimistic bound) |
| **the target's own past** (demand) | actuals up to now, then nothing | recursive: your predictions (compounding); direct / multi-output: only real history goes in, the model learned the drift |

The professional pattern is **direct or multi-output with forecast weather** - no fake inputs, and the weather forecast error is explicit. Recursive is fine for a few steps and for models that are expensive to retrain per horizon.

**One more way, for later:** an *encoder-decoder* (seq2seq) RNN reads the window and then *generates* the next 24 values one at a time, trained with **teacher forcing** (the true previous value is fed in during training) - which is exactly the exposure-bias situation, made explicit and usually managed with scheduled sampling. That's the architecture that becomes machine translation in Module 5.

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 18b — Another use case: two weeks of daily demand - past demand or the weather forecast?
- Same question, new use case: the planning desk wants the next 14 DAYS, one number per day. Daily totals (GWh), 2011-2018 train, 2019 holdout.
- Recursive LSTM with past demand + calendar + temperature. The future temperature column needs a source: actual (perfect forecast), the calendar-day average from the training years (honest, no forecast), or frozen.
- Read the table: day +1 is about 40% better than same-weekday-last-week; by day +14 it only ties. And the temperature source barely matters for this model - it leans on past demand.
- Now drop past demand entirely: demand from temperature and calendar only. Nothing is fed back, so the error is flat across the horizon. With perfect temperatures it wins at day +14 and its slope is ~1; with average temperatures it loses to the naive forecast.
- The lesson: past demand buys you the first few days; beyond a week, a demand forecast is a WEATHER forecast. That's why utilities buy weather data. Compare temperature 7b: the univariate LSTM stayed 17% ahead of persistence at day 14.
-->


## Another use case: two weeks of daily demand

The hourly problem above asked for tomorrow. A planning desk asks a different question: **what will each of the next 14 days look like?** Same data, rolled up to **daily totals (GWh per day)**, trained on 2011-2018 and scored on 2019.

Two ways to build it:

| model | inputs for each day | at forecast time |
| :-- | :-- | :-- |
| **Recursive LSTM** | the past 14 days of demand, temperature and calendar | roll forward 14 times; the demand column fills with its own predictions |
| **No-lag network** | only that day's temperature and calendar | no past demand, nothing fed back, so no compounding |

Both need **temperature for days that haven't happened**. We try three sources: the **actual** temperatures (a perfect forecast, the best case), the **calendar-day average** from the training years (the honest no-forecast case), and the last observed temperature held **frozen**.

In [ ]:
daily = pd.DataFrame({"demand": df["Demand"].resample("D").sum() / 1000.0,       # GWh per day
                      "temp":   df["BDL_tmpf"].resample("D").mean()}).loc["2011":"2019"].dropna()
daily["ds"], daily["dc"] = np.sin(2*np.pi*daily.index.dayofweek/7), np.cos(2*np.pi*daily.index.dayofweek/7)
daily["ys"], daily["yc"] = np.sin(2*np.pi*daily.index.dayofyear/365.25), np.cos(2*np.pi*daily.index.dayofyear/365.25)
dcols = ["temp", "ds", "dc", "ys", "yc", "demand"]                        # demand LAST
d_tr, d_te = daily.loc[:"2018-12-31", dcols], daily.loc["2019-01-01":, dcols]
clim = d_tr.groupby(d_tr.index.dayofyear)["temp"].mean()               # average temperature for each calendar day

dsc = MinMaxScaler().fit(d_tr)                                         # fit on TRAIN only
D_tr, D_te = dsc.transform(d_tr), dsc.transform(d_te)
to_gwh = lambda v: v * (dsc.data_max_[-1] - dsc.data_min_[-1]) + dsc.data_min_[-1]
t_scaled = lambda v: (v - dsc.data_min_[0]) / (dsc.data_max_[0] - dsc.data_min_[0])
print("daily train:", d_tr.shape, "| holdout 2019:", d_te.shape, f"| mean 2019 demand {d_te['demand'].mean():.1f} GWh/day")

In [ ]:
d_lookback, d_horizon = 14, 14                                                # two weeks in, two weeks out
keras.utils.set_random_seed(5509)
Xd = np.stack([D_tr[i:i+d_lookback] for i in range(len(D_tr) - d_lookback)]); yd = D_tr[d_lookback:, -1]
daily_lstm = Sequential([LSTM(32, input_shape=Xd.shape[1:]), Dense(1)])
daily_lstm.compile(optimizer="adam", loss="mse")
daily_lstm.fit(Xd, yd, epochs=60, batch_size=32, validation_split=0.2, verbose=0,
               callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)])

n_o = len(D_te) - d_lookback - d_horizon + 1
d_dates = d_te.index
d_truth = to_gwh(np.stack([D_te[i+d_lookback:i+d_lookback+d_horizon, -1] for i in range(n_o)]))

def roll_daily(temp_source):
    window = np.stack([D_te[i:i+d_lookback] for i in range(n_o)])
    frozen = window[:, -1, 0].copy()                                   # last observed temperature
    out = np.zeros((n_o, d_horizon))
    for h in range(d_horizon):
        p = daily_lstm.predict(window, verbose=0).ravel(); out[:, h] = p
        nxt = np.stack([D_te[i+d_lookback+h] for i in range(n_o)]).copy()  # calendar columns: known exactly
        if temp_source == "average":
            nxt[:, 0] = [t_scaled(clim.get(d_dates[i+d_lookback+h].dayofyear, clim.mean())) for i in range(n_o)]
        elif temp_source == "frozen":
            nxt[:, 0] = frozen
        nxt[:, -1] = p                                                  # demand: our own prediction
        window = np.concatenate([window[:, 1:], nxt[:, None]], axis=1)
    return to_gwh(out)

rec_daily = {src: roll_daily(src) for src in ["actual", "average", "frozen"]}
print("recursive LSTM rolled 14 days from", n_o, "origins, three temperature sources")

In [ ]:
# no past demand at all: demand(day) = f(temperature(day), calendar(day))
keras.utils.set_random_seed(5509)
nolag = Sequential([Dense(32, activation="relu", input_shape=(5,)), Dense(16, activation="relu"), Dense(1)])
nolag.compile(optimizer="adam", loss="mse")
nolag.fit(D_tr[:, :5], D_tr[:, -1], epochs=200, batch_size=32, validation_split=0.2, verbose=0,
          callbacks=[EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)])

nolag_daily = {}
for src in ["actual", "average"]:
    Z = D_te[:, :5].copy()
    if src == "average":
        Z[:, 0] = [t_scaled(clim.get(dt.dayofyear, clim.mean())) for dt in d_dates]
    p = to_gwh(nolag.predict(Z, verbose=0).ravel())
    nolag_daily[src] = np.stack([p[i+d_lookback:i+d_lookback+d_horizon] for i in range(n_o)])

dd = d_te["demand"].values
lag = lambda h: 7 if h < 7 else 14                                      # most recent same weekday known at the origin
d_naive = np.array([np.abs(dd[d_lookback+h:d_lookback+h+n_o] - dd[d_lookback+h-lag(h):d_lookback+h-lag(h)+n_o]).mean() for h in range(d_horizon)])
d_pers = np.array([np.abs(dd[d_lookback+h:d_lookback+h+n_o] - dd[d_lookback-1:d_lookback-1+n_o]).mean() for h in range(d_horizon)])

models = {"recursive LSTM, actual temps":   rec_daily["actual"],
          "recursive LSTM, average temps":  rec_daily["average"],
          "recursive LSTM, frozen temp":    rec_daily["frozen"],
          "no past demand, actual temps":   nolag_daily["actual"],
          "no past demand, average temps":  nolag_daily["average"]}
H = [0, 2, 6, 13]
table = pd.DataFrame({k: np.abs(P - d_truth).mean(axis=0)[H] for k, P in models.items()}).T
table.loc["same weekday, latest week"] = d_naive[H]
table.loc["last day held"] = d_pers[H]
table.columns = [f"day +{h+1}" for h in H]
table["slope at day +14"] = [np.polyfit(d_truth[:, 13], P[:, 13], 1)[0] for P in models.values()] + [np.nan, np.nan]
table.round(2)

In [ ]:
days = np.arange(1, d_horizon + 1)
plt.figure(figsize=(9, 4))
for k, style in [("recursive LSTM, average temps", "o-"), ("recursive LSTM, actual temps", ".--"),
                 ("no past demand, actual temps", "s-"), ("no past demand, average temps", "s:")]:
    plt.plot(days, np.abs(models[k] - d_truth).mean(axis=0), style, label=k)
plt.plot(days, d_naive, color="gray", lw=2, label="same weekday, latest week")
plt.xlabel("days ahead"); plt.ylabel("2019 MAE (GWh/day)"); plt.title("Two weeks of daily demand: who wins, and when?")
plt.legend(fontsize=8); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)
for a, k in zip(ax, ["recursive LSTM, average temps", "no past demand, actual temps"]):
    a.scatter(d_truth[:, 13], models[k][:, 13], s=9, alpha=0.5)
    lim = [d_truth.min(), d_truth.max()]; a.plot(lim, lim, color="red", lw=1.2)
    a.set_title(f"day +14, {k}"); a.set_xlabel("actual (GWh/day)")
ax[0].set_ylabel("predicted (GWh/day)"); plt.tight_layout(); plt.show()

### Two weeks of daily demand, as time series

The scoreboard averages over 338 forecast origins. Here is what those forecasts look like across **all of 2019**, lined up by the day they were for, at three lead times: made **1**, **7** and **14** days earlier.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(13, 9.5), sharex=True, sharey=True)
for a, h in zip(ax, [1, 7, 14]):
    when = d_dates[d_lookback + h - 1 : d_lookback + h - 1 + n_o]           # the day each forecast is for
    a.plot(when, d_truth[:, h-1], color="black", lw=1.6, label="actual")
    a.plot(when, models["recursive LSTM, average temps"][:, h-1], lw=1.1, label="recursive LSTM, average temps")
    a.plot(when, models["no past demand, actual temps"][:, h-1], lw=1.1, label="no past demand, actual temps")
    a.plot(when, models["no past demand, average temps"][:, h-1], lw=1.1, ls="--", label="no past demand, average temps")
    a.set_title(f"forecasts made {h} day{'s' if h > 1 else ''} earlier"); a.set_ylabel("GWh/day")
ax[0].legend(ncol=2, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

**What to look for.** At 1 day the recursive LSTM rides the black line. At 7 and 14 days it smooths into a seasonal curve, and like every model here it sits **above** the black line for most of the year. The average-temperature network is the **same curve every time you run it** (the calendar and the climate average never change), which is why it can't see a heat wave coming. Only the network given the actual temperatures follows the spikes, and it does so equally well at every lead time.

In [ ]:
# the trend view: monthly averages of the 14-day-ahead forecasts
when = d_dates[d_lookback + 13 : d_lookback + 13 + n_o]
monthly = pd.DataFrame({"actual": d_truth[:, 13],
                        "recursive LSTM, average temps": models["recursive LSTM, average temps"][:, 13],
                        "no past demand, actual temps": models["no past demand, actual temps"][:, 13],
                        "no past demand, average temps": models["no past demand, average temps"][:, 13]},
                       index=when).resample("MS").mean()
ax = monthly.plot(figsize=(11, 3.8), marker="o", style=["k-", "-", "-", "--"], lw=1.5,
                  title="Monthly average of the forecasts made 14 days earlier (GWh/day)")
ax.set_xlabel(""); ax.set_ylabel("GWh/day"); plt.show()
(monthly.drop(columns="actual").sub(monthly["actual"], axis=0)).round(1).rename(columns=lambda c: c + " - actual")

**The trend view.** Averaged by month, every model gets the seasonal shape (winter and summer peaks, spring and fall valleys). The table shows the **monthly bias**: a positive number means the forecasts ran high. **Almost every number is positive**, up to about 10 GWh/day in September and October, even for the model that knows the actual temperatures. That's a trend the models never saw: 2019 demand sat below the 2011-2018 pattern they learned (efficiency gains and rooftop solar are the usual suspects). No lead time or architecture fixes that; retraining on recent years, or adding a trend term, does.

**What the table says.**

- **Past demand buys you the first few days.** The recursive LSTM is far ahead at day +1 and still competitive at day +7.
- **By day +14 it only ties the naive forecast**, and the temperature source barely changes it: the model leans on its own recent demand, which is now mostly its own guesses.
- **The no-past-demand network has a flat error**, because nothing is fed back. With the actual temperatures it's the best model at day +14, and its predictions line up with the truth (slope near 1). With average temperatures it loses to the naive forecast.

**The lesson:** beyond about a week, a demand forecast is really a **weather forecast** plus the calendar. That's why utilities pay for weather data, and why the professional two-week model is "temperature forecast in, demand out."

**Compare the temperature notebook (video 7b):** a univariate LSTM rolled 14 days ahead stayed 17% ahead of persistence. A smooth, mean-reverting series is easy to roll forward; a demand series driven by weather is not.

## Save the model and use it again

In [ ]:
mimo.save('Multi_Step_Forecasting_Strategies_mimo.keras')
reloaded = load_model('Multi_Step_Forecasting_Strategies_mimo.keras')
print("reloaded model reproduces the forecast:", np.allclose(mimo.predict(Xm_te[:5], verbose=0), reloaded.predict(Xm_te[:5], verbose=0)))

## On your own

- Train all 24 direct models (a loop, ~10 minutes on CPU) and plot the full direct curve. Does it beat multi-output at every horizon?
- Recursive with a **week** of history (`n_steps = 168`). Does a longer window slow the error growth?
- Replace the frozen weather with *yesterday's weather at the same hour* (a poor person's forecast). Where does it land between the two recursive curves?
- Add a `holiday` column (`pandas.tseries.holiday.USFederalHolidayCalendar`) - a calendar covariate you know perfectly in advance.